# Vague-map direct-bin tester

This focused test starts at **Planning** with `grabbed=True`. It verifies that vague-map navigation overrides normal bin searching, then drives only toward the encoded `bin_docking_pose`. With **Force Tag missing** enabled, it proves the map behavior without relying on AprilTag visibility.

Coordinate convention: `+Y` is the initial forward direction, `+X` is right, and positive heading rotates clockwise from `+Y` toward `+X`. If the robot starts left of the bin, its initial `x_m` should be smaller than the bin docking `x_m`, so the expected first motion is `right`. Reverse those values to test the left-turn case.

Edit `tuning_tools/vague_map_test_parameters.json`, click **Reload JSON**, and check the preview assertion before enabling real motion. `bin_marker_position` records the physical Tag location; `bin_docking_pose` is the point the navigator actually targets.

For forward calibration, set high speed and pulse duration in the JSON, run one pulse, measure travel distance, enter it as `measured_distance_m`, and reload. The reported factor is a suggestion for `vague_map.odometry.linear_meters_per_speed_second`; production `empirical_parameters.json` is never modified. Turning uses the existing measured `base_turn_response`: at a selected speed, angle is assumed proportional to time, and headings are normalized to `[-pi, pi)`.

Safety: real base and real camera are disabled by default. The run stops when the coarse map destination is reached or the Tag is detected; it does not continue into searching, visual docking, or arm actions. Keep the physical stop path clear and use **STOP BASE** at any time.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
candidates = [here, here.parent]
project_root = next((path for path in candidates if (path / 'config.json').exists()), None)
if project_root is None:
    raise RuntimeError('Run from the repository root or tuning_tools directory; config.json was not found')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tuning_tools.vague_map_tester import build_ui
panel = build_ui(str(project_root / 'tuning_tools' / 'vague_map_test_parameters.json'))
